In [0]:
%run "/Users/gresahasani19@gmail.com/urban-mobility-lakehouse/src/quality/rules"


In [0]:
%run "/Users/gresahasani19@gmail.com/urban-mobility-lakehouse/src/quality/validator"

In [0]:
from pyspark.sql import functions as F

bronze_stream = spark.readStream.format("delta").table("urban_mobility.bronze.trip_events")

typed_stream = (
    bronze_stream
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
    .withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))
    .withColumn("trip_status", F.upper(F.trim(F.col("trip_status"))))
    .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
)

deduped_stream = (
    typed_stream
    .withWatermark("event_timestamp", "2 hours")
    .dropDuplicates(["event_id"])
)

In [0]:
def process_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    rules = get_trip_event_rules()
    valid_df, quarantine_df, dq_results_df, _, valid_count, invalid_count = run_validation(
        batch_df, rules, "bronze.trip_events"
    )

    valid_df.write.format("delta").mode("append").saveAsTable("urban_mobility.silver.trip_events_clean")
    quarantine_df.write.format("delta").mode("append").saveAsTable("urban_mobility.quarantine.invalid_trip_events")
    dq_results_df.write.format("delta").mode("append").saveAsTable("urban_mobility.monitoring.data_quality_results")

    print(f"Batch {batch_id}: valid={valid_count}, quarantine={invalid_count}")

checkpoint_path = "/Volumes/urban_mobility/bronze/checkpoints/clean_trip_events/"

query = (
    deduped_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("silver.trip_events_clean rows:", spark.table("urban_mobility.silver.trip_events_clean").count())
print("quarantine.invalid_trip_events rows:", spark.table("urban_mobility.quarantine.invalid_trip_events").count())
print("monitoring.data_quality_results rows:", spark.table("urban_mobility.monitoring.data_quality_results").count())